# 최종 취약지수 및 DBSCAN 권역화

- 목적: 이미 산출한 격자별 최종취약지수와 하위지표 점수를 기준으로 취약격자를 DBSCAN으로 권역화함.
- 분석 범위: K-means 유형화는 수행하지 않고, 권역 경계 생성과 권역별 기본 특성 집계까지만 수행함.
- 산출물 최소화: 별도 고립격자 파일, 파라미터 비교 파일, 공간 중간파일은 생성하지 않음.
- 핵심 산출물: 격자별 최종취약지수 CSV, DBSCAN 대상 취약격자 CSV, DBSCAN 취약권역 CSV.


## 1. 분석 환경 및 경로 설정

- 프로젝트 경로를 기준으로 접근성 산출물과 최종 취약지수 산출물 경로를 설정함.
- 최종 산출물은 `notebooks/dashboard/OUTPUT/vulnerability_index`에 저장함.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import DBSCAN

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

BASE_PATH = Path.cwd().resolve()

if BASE_PATH.name == "dashboard":
    PROJECT_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    PROJECT_PATH = BASE_PATH.parent
elif (BASE_PATH / "notebooks").exists():
    PROJECT_PATH = BASE_PATH
else:
    PROJECT_PATH = BASE_PATH

NOTEBOOK_PATH = PROJECT_PATH / "notebooks"
ACCESS_OUTPUT_PATH = NOTEBOOK_PATH / "access" / "OUTPUT"
DASHBOARD_PATH = NOTEBOOK_PATH / "dashboard"
OUTPUT_PATH = DASHBOARD_PATH / "OUTPUT" / "vulnerability_index"
DOCS_PATH = DASHBOARD_PATH / "docs"
IMAGE_PATH = DASHBOARD_PATH / "IMAGE"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
DOCS_PATH.mkdir(parents=True, exist_ok=True)
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

print("PROJECT_PATH:", PROJECT_PATH)
print("ACCESS_OUTPUT_PATH:", ACCESS_OUTPUT_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)


## 2. 최종 취약지수 불러오기

- 입력자료: `notebooks/access/OUTPUT/final_vulnerability_index/종합문화취약지수_선호반영_H3SFCA.csv`를 사용함.
- 처리 방식: 이미 산출된 최종취약점수와 하위지표 점수를 DBSCAN 입력 형태로 정리함.
- 재계산 제외: H3SFCA, 다양성, 장애인·노인 E2SFCA 원천 계산은 다시 수행하지 않음.


In [ ]:
final_index_path = ACCESS_OUTPUT_PATH / "final_vulnerability_index" / "종합문화취약지수_선호반영_H3SFCA.csv"

if not final_index_path.exists():
    raise FileNotFoundError(f"최종 취약지수 파일이 없습니다: {final_index_path}")

base = pd.read_csv(final_index_path)

rename_cols = {
    "문화누리대상자_추정_인구수": "문화누리대상자_추정인구수",
    "분석대상": "분석대상여부",
    "주요취약원인1": "주요취약원인_1",
    "주요취약원인2": "주요취약원인_2"
}
base = base.rename(columns={key: value for key, value in rename_cols.items() if key in base.columns})

required_input_cols = [
    "GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y",
    "문화누리대상자_추정인구수", "분석대상여부", "최종취약지수_백분위",
    "시설접근성취약점수_z", "다양성취약점수_z", "노인편의취약점수_z", "장애인친화취약점수_z",
    "장애인_수요인구수", "노령인구_수요인구수"
]
missing_input_cols = [col for col in required_input_cols if col not in base.columns]
if missing_input_cols:
    raise KeyError(f"최종 취약지수 입력에 필요한 컬럼이 없습니다: {missing_input_cols}")

num_cols = [
    "중심점_x", "중심점_y", "추정_인구수", "문화누리대상자_추정인구수",
    "최종취약지수_백분위", "시설접근성취약점수_z", "다양성취약점수_z",
    "노인편의취약점수_z", "장애인친화취약점수_z",
    "장애인_수요인구수", "노령인구_수요인구수"
]
num_cols = [col for col in num_cols if col in base.columns]
base[num_cols] = base[num_cols].apply(pd.to_numeric, errors="coerce")

base["분석대상여부"] = base["분석대상여부"].fillna(False).astype(bool)
base["최종취약지수"] = base["최종취약지수_백분위"]

score_source_map = {
    "시설분류_접근성취약도": "시설접근성취약점수_z",
    "문화다양성부족도": "다양성취약점수_z",
    "장애인친화시설_접근성취약도": "장애인친화취약점수_z",
    "노인편의서비스_접근성취약도": "노인편의취약점수_z"
}

def make_percentile_score(series, mask):
    score = pd.Series(np.nan, index=series.index, dtype="float64")
    score.loc[mask] = series.loc[mask].rank(pct=True, method="average") * 100
    return score

for new_col, source_col in score_source_map.items():
    valid_mask = base["분석대상여부"] & base[source_col].notna()
    base[new_col] = make_percentile_score(base[source_col], valid_mask)

base["대상자수_분위"] = 0.0
target_mask = base["분석대상여부"] & base["문화누리대상자_추정인구수"].notna()
base.loc[target_mask, "대상자수_분위"] = base.loc[target_mask, "문화누리대상자_추정인구수"].rank(pct=True, method="average") * 100
base["우선지원지수"] = base["최종취약지수"].fillna(0) * base["대상자수_분위"] / 100

cause_cols = [
    "시설분류_접근성취약도",
    "문화다양성부족도",
    "장애인친화시설_접근성취약도",
    "노인편의서비스_접근성취약도"
]

cause_label = {
    "시설분류_접근성취약도": "시설분류 접근성 부족",
    "문화다양성부족도": "문화시설 다양성 부족",
    "장애인친화시설_접근성취약도": "장애인친화시설 접근성 부족",
    "노인편의서비스_접근성취약도": "노인편의서비스 접근성 부족"
}

def get_top_causes(row, n=2):
    values = row[cause_cols].sort_values(ascending=False)
    return [cause_label[idx] for idx in values.index[:n]]

need_cause = ("주요취약원인_1" not in base.columns) or ("주요취약원인_2" not in base.columns)
if need_cause:
    cause_values = base.apply(get_top_causes, axis=1, result_type="expand")
    base["주요취약원인_1"] = cause_values[0]
    base["주요취약원인_2"] = cause_values[1]

print("최종 취약지수 입력자료")
print("- 파일:", final_index_path)
print("- 행/열:", base.shape)
print("- 지수모형:", base["지수모형"].dropna().unique().tolist() if "지수모형" in base.columns else "확인불가")


## 3. 전처리 및 점수 품질 점검

- 컬럼 정리: 기존 최종취약지수 파일의 컬럼명을 DBSCAN 분석용 이름으로 통일함.
- 점수 정리: 최종취약지수는 기존 백분위 점수를 사용하고, 하위지표 z점수는 분석대상 내 백분위 점수로 변환함.
- 인구 처리: 취약점수에는 인구수를 추가하지 않고, 권역 요약과 우선지원지수 계산에만 사용함.
- 품질 점검: 중복 격자, 좌표 결측, 점수 결측, 대상 인구 분포를 확인함.


In [ ]:
quality_check = {
    "전체격자수": len(base),
    "분석대상격자수": int(base["분석대상여부"].sum()),
    "중복_GRID_CD수": int(base["GRID_CD"].duplicated().sum()),
    "좌표결측격자수": int(base[["중심점_x", "중심점_y"]].isna().any(axis=1).sum()),
    "최종취약지수결측수": int(base.loc[base["분석대상여부"], "최종취약지수"].isna().sum()),
    "문화누리대상자_보유격자수": int((base["문화누리대상자_추정인구수"] > 0).sum()),
    "장애인수요_보유격자수": int((base["장애인_수요인구수"] > 0).sum()),
    "노령인구수요_보유격자수": int((base["노령인구_수요인구수"] > 0).sum())
}

print("전처리 품질 점검")
for key, value in quality_check.items():
    print(f"- {key}: {value:,}")

print()
print("취약등급별 격자 수")
print(base["취약등급"].value_counts(dropna=False).to_string())

print()
print("분석대상 최종취약지수 분포")
print(base.loc[base["분석대상여부"], "최종취약지수"].describe().to_string())

print()
print("하위지표 백분위 점수 분포")
display(base.loc[base["분석대상여부"], cause_cols].describe().T)

print()
print("상위 취약 격자 예시")
display(base.loc[base["분석대상여부"]].sort_values("최종취약지수", ascending=False).head(10))


## 4. DBSCAN 취약권역 생성

- 분석 기준: 종합취약지수와 4개 하위지표를 각각 별도 필터로 권역화함.
- 취약격자 기준: 각 지표별 관련 인구가 있는 격자 중 취약점수 상위 10%를 사용함.
- DBSCAN 설정: 100m 격자 구조를 고려해 `eps=150m`, `min_samples=4`를 적용함.
- 저장 방식: 권역화 대상 격자와 권역 요약 CSV만 저장하고, 고립 취약격자는 별도 파일 없이 `DBSCAN_label=-1`로 구분함.
- 점검 항목: 좌표 결측, 중복 격자, 지표별 대상 격자 수, 권역 수, 고립격자 비율을 확인함.


In [ ]:
final_grid = base.copy()

DBSCAN_EPS = 150
DBSCAN_MIN_SAMPLES = 4
VULNERABLE_QUANTILE = 0.90
GRID_AREA_M2 = 100 * 100

score_configs = [
    {
        "권역유형": "종합취약",
        "권역접두어": "VUL",
        "점수컬럼": "최종취약지수",
        "대상컬럼": "문화누리대상자_추정인구수",
        "대상조건": final_grid["분석대상여부"]
    },
    {
        "권역유형": "시설접근성취약",
        "권역접두어": "FAC",
        "점수컬럼": "시설분류_접근성취약도",
        "대상컬럼": "문화누리대상자_추정인구수",
        "대상조건": final_grid["분석대상여부"]
    },
    {
        "권역유형": "문화다양성취약",
        "권역접두어": "DIV",
        "점수컬럼": "문화다양성부족도",
        "대상컬럼": "문화누리대상자_추정인구수",
        "대상조건": final_grid["분석대상여부"]
    },
    {
        "권역유형": "장애인친화취약",
        "권역접두어": "DIS",
        "점수컬럼": "장애인친화시설_접근성취약도",
        "대상컬럼": "장애인_수요인구수",
        "대상조건": final_grid["장애인_수요인구수"] > 0
    },
    {
        "권역유형": "노인편의취약",
        "권역접두어": "ELD",
        "점수컬럼": "노인편의서비스_접근성취약도",
        "대상컬럼": "노령인구_수요인구수",
        "대상조건": final_grid["노령인구_수요인구수"] > 0
    }
]

required_dbscan_cols = [
    "GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y",
    "문화누리대상자_추정인구수", "장애인_수요인구수", "노령인구_수요인구수",
    "최종취약지수", "시설분류_접근성취약도", "문화다양성부족도",
    "장애인친화시설_접근성취약도", "노인편의서비스_접근성취약도"
]

missing_cols = [col for col in required_dbscan_cols if col not in final_grid.columns]
if missing_cols:
    raise KeyError(f"DBSCAN 입력에 필요한 컬럼이 없습니다: {missing_cols}")

quality_summary = {
    "전체격자수": len(final_grid),
    "중복_GRID_CD수": int(final_grid["GRID_CD"].duplicated().sum()),
    "좌표결측격자수": int(final_grid[["중심점_x", "중심점_y"]].isna().any(axis=1).sum()),
    "분석대상격자수": int(final_grid["분석대상여부"].sum())
}

if quality_summary["중복_GRID_CD수"] > 0:
    raise ValueError("GRID_CD 중복이 있어 권역화 전에 확인이 필요합니다.")

if quality_summary["좌표결측격자수"] > 0:
    raise ValueError("중심점 좌표 결측이 있어 DBSCAN을 수행할 수 없습니다.")

cluster_grid_list = []
cluster_rows = []
filter_summary_rows = []

for config in score_configs:
    area_type = config["권역유형"]
    prefix = config["권역접두어"]
    score_col = config["점수컬럼"]
    population_col = config["대상컬럼"]

    target_mask = config["대상조건"] & final_grid[score_col].notna()
    target_grid = final_grid[target_mask].copy()

    if len(target_grid) == 0:
        filter_summary_rows.append({
            "권역유형": area_type,
            "기준점수컬럼": score_col,
            "취약선정기준": None,
            "분석대상격자수": 0,
            "DBSCAN대상격자수": 0,
            "권역포함격자수": 0,
            "고립격자수": 0,
            "취약권역수": 0,
            "고립격자비율": 0
        })
        continue

    threshold = target_grid[score_col].quantile(VULNERABLE_QUANTILE)
    vulnerable_grid = target_grid[target_grid[score_col] >= threshold].copy()
    vulnerable_grid["권역유형"] = area_type
    vulnerable_grid["기준점수컬럼"] = score_col
    vulnerable_grid["기준취약점수"] = vulnerable_grid[score_col]
    vulnerable_grid["관련인구컬럼"] = population_col
    vulnerable_grid["관련인구수"] = vulnerable_grid[population_col]
    vulnerable_grid["취약선정기준"] = threshold

    coords = vulnerable_grid[["중심점_x", "중심점_y"]].to_numpy()

    if len(vulnerable_grid) >= DBSCAN_MIN_SAMPLES:
        dbscan = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
        vulnerable_grid["DBSCAN_label"] = dbscan.fit_predict(coords)
    else:
        vulnerable_grid["DBSCAN_label"] = -1

    vulnerable_grid["취약권역_ID"] = np.where(
        vulnerable_grid["DBSCAN_label"] >= 0,
        prefix + "_" + (vulnerable_grid["DBSCAN_label"] + 1).astype(str).str.zfill(3),
        "고립취약격자"
    )

    clustered = vulnerable_grid[vulnerable_grid["DBSCAN_label"] >= 0].copy()
    isolated = vulnerable_grid[vulnerable_grid["DBSCAN_label"] < 0].copy()

    filter_summary_rows.append({
        "권역유형": area_type,
        "기준점수컬럼": score_col,
        "취약선정기준": threshold,
        "분석대상격자수": len(target_grid),
        "DBSCAN대상격자수": len(vulnerable_grid),
        "권역포함격자수": len(clustered),
        "고립격자수": len(isolated),
        "취약권역수": clustered["취약권역_ID"].nunique(),
        "고립격자비율": len(isolated) / len(vulnerable_grid) if len(vulnerable_grid) > 0 else 0
    })

    cluster_grid_list.append(vulnerable_grid)

    for cluster_id, temp in clustered.groupby("취약권역_ID"):
        weight = temp[population_col].replace(0, np.nan)
        if weight.notna().sum() > 0:
            weighted_score = np.average(temp.loc[weight.notna(), score_col], weights=weight.dropna())
            weighted_final = np.average(temp.loc[weight.notna(), "최종취약지수"], weights=weight.dropna())
        else:
            weighted_score = temp[score_col].mean()
            weighted_final = temp["최종취약지수"].mean()

        component_mean = temp[cause_cols].mean().sort_values(ascending=False)
        top_causes = [cause_label[idx] for idx in component_mean.index[:2]]

        cluster_rows.append({
            "권역유형": area_type,
            "취약권역_ID": cluster_id,
            "기준점수컬럼": score_col,
            "취약선정기준": threshold,
            "포함_취약격자수": len(temp),
            "권역면적_m2": len(temp) * GRID_AREA_M2,
            "권역중심_x": temp["중심점_x"].mean(),
            "권역중심_y": temp["중심점_y"].mean(),
            "관련인구컬럼": population_col,
            "관련인구수": temp[population_col].sum(),
            "문화누리대상자_추정인구수": temp["문화누리대상자_추정인구수"].sum(),
            "장애인_수요인구수": temp["장애인_수요인구수"].sum(),
            "노령인구_수요인구수": temp["노령인구_수요인구수"].sum(),
            "평균_기준취약점수": temp[score_col].mean(),
            "최고_기준취약점수": temp[score_col].max(),
            "수요가중_기준취약점수": weighted_score,
            "평균_최종취약지수": temp["최종취약지수"].mean(),
            "수요가중_최종취약지수": weighted_final,
            "평균_우선지원지수": temp["우선지원지수"].mean(),
            "평균_시설분류_접근성취약도": temp["시설분류_접근성취약도"].mean(),
            "평균_문화다양성부족도": temp["문화다양성부족도"].mean(),
            "평균_장애인친화시설_접근성취약도": temp["장애인친화시설_접근성취약도"].mean(),
            "평균_노인편의서비스_접근성취약도": temp["노인편의서비스_접근성취약도"].mean(),
            "주요_시군구": temp["시군구"].mode().iloc[0] if len(temp["시군구"].mode()) > 0 else None,
            "주요_행정동": temp["행정동"].mode().iloc[0] if len(temp["행정동"].mode()) > 0 else None,
            "주요취약원인_1": top_causes[0],
            "주요취약원인_2": top_causes[1]
        })

vulnerable_grid = pd.concat(cluster_grid_list, ignore_index=True) if cluster_grid_list else final_grid.iloc[0:0].copy()
cluster_gdf = pd.DataFrame(cluster_rows)
filter_summary = pd.DataFrame(filter_summary_rows)

if len(cluster_gdf) > 0:
    cluster_gdf["취약권역등급"] = "일반취약권역"
    for area_type, temp in cluster_gdf.groupby("권역유형"):
        q90_cluster = temp["수요가중_기준취약점수"].quantile(0.90)
        q80_cluster = temp["수요가중_기준취약점수"].quantile(0.80)
        area_mask = cluster_gdf["권역유형"] == area_type
        cluster_gdf.loc[area_mask & (cluster_gdf["수요가중_기준취약점수"] >= q90_cluster), "취약권역등급"] = "최우선취약권역"
        cluster_gdf.loc[
            area_mask
            & (cluster_gdf["수요가중_기준취약점수"] < q90_cluster)
            & (cluster_gdf["수요가중_기준취약점수"] >= q80_cluster),
            "취약권역등급"
        ] = "우선취약권역"

print("전처리 품질 점검")
for key, value in quality_summary.items():
    print(f"- {key}: {value:,}")

print()
print("DBSCAN 필터별 결과")
display(filter_summary)

if len(cluster_gdf) > 0:
    print()
    print("권역 규모 요약")
    display(
        cluster_gdf
        .groupby("권역유형")
        .agg(
            취약권역수=("취약권역_ID", "count"),
            평균격자수=("포함_취약격자수", "mean"),
            최대격자수=("포함_취약격자수", "max"),
            평균기준취약점수=("평균_기준취약점수", "mean"),
            관련인구수=("관련인구수", "sum")
        )
        .reset_index()
    )

    print()
    print("권역별 상위 예시")
    display(
        cluster_gdf
        .sort_values(["권역유형", "수요가중_기준취약점수"], ascending=[True, False])
        .groupby("권역유형")
        .head(3)
    )


## 5. 산출물 저장

- 저장 원칙: 대시보드와 후속 분석에 필요한 핵심 CSV만 저장함.
- 격자 산출물: 최종취약점수, 하위지표 점수, 대상 인구, 주요 취약 원인만 저장함.
- DBSCAN 산출물: 권역화 대상 격자의 권역 ID와 기준점수 확인에 필요한 컬럼만 저장함.
- 제외 산출물: 고립 취약격자 별도 파일, GPKG 공간파일, 파라미터 비교 파일, 중간 점검 파일은 생성하지 않음.


In [ ]:
final_csv_path = OUTPUT_PATH / "격자별_최종취약지수.csv"
vulnerable_grid_csv_path = OUTPUT_PATH / "DBSCAN_취약격자.csv"
cluster_csv_path = OUTPUT_PATH / "DBSCAN_취약권역.csv"

final_output_cols = [
    "GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y", "추정_인구수",
    "문화누리대상자_추정인구수", "장애인_수요인구수", "노령인구_수요인구수",
    "분석대상여부", "지수모형", "취약등급",
    "최종취약지수", "시설분류_접근성취약도", "문화다양성부족도",
    "장애인친화시설_접근성취약도", "노인편의서비스_접근성취약도",
    "대상자수_분위", "우선지원지수", "주요취약원인_1", "주요취약원인_2",
    "시설접근성_최취약중분류", "diversity_type_n", "reachable_store_n",
    "접근가능중분류수", "평균접근가능가맹점수",
    "장애인친화시설_도보_E2SFCA", "장애인친화시설_대중교통_E2SFCA",
    "장애인친화시설_통합접근성", "장애인친화시설_통합도달가맹점수",
    "장애인친화시설_도달가능중분류", "장애인친화시설_부족중분류",
    "노인편의서비스_도보_E2SFCA", "노인편의서비스_대중교통_E2SFCA",
    "노인편의서비스_통합접근성", "노인편의서비스_통합도달가맹점수",
    "노인편의서비스_도달가능중분류", "노인편의서비스_부족중분류"
]
final_output_cols = [col for col in final_output_cols if col in final_grid.columns]

vulnerable_grid_output_cols = [
    "권역유형", "GRID_CD", "취약권역_ID", "DBSCAN_label",
    "기준점수컬럼", "기준취약점수", "취약선정기준",
    "관련인구컬럼", "관련인구수",
    "시군구", "행정동", "중심점_x", "중심점_y",
    "문화누리대상자_추정인구수", "장애인_수요인구수", "노령인구_수요인구수",
    "최종취약지수", "시설분류_접근성취약도", "문화다양성부족도",
    "장애인친화시설_접근성취약도", "노인편의서비스_접근성취약도",
    "취약등급", "주요취약원인_1", "주요취약원인_2"
]
vulnerable_grid_output_cols = [col for col in vulnerable_grid_output_cols if col in vulnerable_grid.columns]

final_grid[final_output_cols].to_csv(final_csv_path, index=False, encoding="utf-8-sig")
vulnerable_grid[vulnerable_grid_output_cols].to_csv(vulnerable_grid_csv_path, index=False, encoding="utf-8-sig")
cluster_gdf.to_csv(cluster_csv_path, index=False, encoding="utf-8-sig")

print("저장 완료")
print(final_csv_path)
print(vulnerable_grid_csv_path)
print(cluster_csv_path)
print("격자 저장 컬럼 수:", len(final_output_cols))
print("취약격자 저장 컬럼 수:", len(vulnerable_grid_output_cols))
print("권역 저장 컬럼 수:", len(cluster_gdf.columns))


## 6. DBSCAN 취약 권역 지도

- 목적: 최종 대시보드와 보고서에 사용할 DBSCAN 취약권역 지도를 생성함.
- 표시 방식: 지표별 권역 포함 격자를 단일 강조색으로 표시하고, 취약점수 컬러바는 사용하지 않음.
- 저장 위치: `notebooks/dashboard/IMAGE/vulnerability_index/dbscan_vulnerable_regions_top10.png`


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

try:
    import geopandas as gpd
except ImportError:
    gpd = None

STATIC_FONT_DIR = PROJECT_PATH / "analysis_table" / "image" / "_fonts"
MEDIUM_FONT_PATH = STATIC_FONT_DIR / "NotoSansKR-Medium.ttf"
BOLD_FONT_PATH = STATIC_FONT_DIR / "NotoSansKR-Bold.ttf"
FONT_PATH = Path("C:/Windows/Fonts/NotoSansKR-VF.ttf")

if MEDIUM_FONT_PATH.exists():
    fm.fontManager.addfont(str(MEDIUM_FONT_PATH))
    BODY_FONT = fm.FontProperties(fname=str(MEDIUM_FONT_PATH))
    plt.rcParams["font.family"] = BODY_FONT.get_name()
else:
    BODY_FONT = None
    if FONT_PATH.exists():
        fm.fontManager.addfont(str(FONT_PATH))
        plt.rcParams["font.family"] = "Noto Sans KR"
    else:
        plt.rcParams["font.family"] = "Malgun Gothic"

if BOLD_FONT_PATH.exists():
    fm.fontManager.addfont(str(BOLD_FONT_PATH))
    TITLE_FONT = fm.FontProperties(fname=str(BOLD_FONT_PATH))
elif FONT_PATH.exists():
    TITLE_FONT = fm.FontProperties(family="Noto Sans KR", weight="bold")
else:
    TITLE_FONT = None

plt.rcParams.update({
    "axes.unicode_minus": False,
    "figure.facecolor": "#FBF6EF",
    "axes.facecolor": "#FBF6EF",
    "savefig.facecolor": "#FBF6EF",
    "text.color": "#2A211D",
    "axes.titleweight": "bold",
    "figure.titleweight": "bold",
})

THEME_BG = "#FBF6EF"
THEME_DARK = "#2A211D"
THEME_LIGHT_GRAY = "#C9C3BE"
THEME_AXIS = "#D8D2CA"
THEME_ORANGE = "#F46B2F"
DBSCAN_IMAGE_PATH = IMAGE_PATH / "vulnerability_index"
DBSCAN_IMAGE_PATH.mkdir(parents=True, exist_ok=True)

DBSCAN_MAP_TYPES = [
    "종합취약",
    "시설접근성취약",
    "문화다양성취약",
    "장애인친화취약",
    "노인편의취약",
]

TITLE_KWARGS = {"fontproperties": TITLE_FONT} if TITLE_FONT is not None else {"fontweight": "bold"}


def load_seoul_boundary_for_dbscan():
    boundary_path = PROJECT_PATH / "analysis_table" / "data" / "output" / "서울시_시군구_행정동_경계.gpkg"
    if gpd is None or not boundary_path.exists():
        return None, None

    dong_boundary = gpd.read_file(boundary_path)
    if dong_boundary.crs is not None:
        dong_boundary = dong_boundary.to_crs(epsg=5179)

    gu_boundary = dong_boundary[["시군구", "geometry"]].dissolve(by="시군구", as_index=False)
    seoul_boundary = dong_boundary[["geometry"]].dissolve()
    return gu_boundary, seoul_boundary


def draw_dbscan_boundary(ax, gu_boundary, seoul_boundary):
    if gu_boundary is None or seoul_boundary is None:
        return
    gu_boundary.boundary.plot(ax=ax, color=THEME_AXIS, linewidth=0.35, zorder=1)
    seoul_boundary.boundary.plot(ax=ax, color=THEME_DARK, linewidth=0.85, zorder=4)


def set_dbscan_map_extent(ax, data, seoul_boundary=None):
    if seoul_boundary is not None:
        minx, miny, maxx, maxy = seoul_boundary.total_bounds
    else:
        minx, maxx = data["중심점_x"].min(), data["중심점_x"].max()
        miny, maxy = data["중심점_y"].min(), data["중심점_y"].max()

    x_pad = (maxx - minx) * 0.025
    y_pad = (maxy - miny) * 0.025
    ax.set_xlim(minx - x_pad, maxx + x_pad)
    ax.set_ylim(miny - y_pad, maxy + y_pad)
    ax.set_aspect("equal")
    ax.set_facecolor(THEME_BG)
    ax.axis("off")


dbscan_map_grid = vulnerable_grid[vulnerable_grid["DBSCAN_label"].ge(0)].copy()

gu_boundary, seoul_boundary = load_seoul_boundary_for_dbscan()

fig, axes = plt.subplots(
    2,
    3,
    figsize=(13, 8.6),
    dpi=180,
    constrained_layout=False,
    facecolor=THEME_BG,
)
axes = axes.ravel()
fig.subplots_adjust(left=0.02, right=0.98, bottom=0.035, top=0.86, wspace=0.06, hspace=0.30)

for ax, region_type in zip(axes[:len(DBSCAN_MAP_TYPES)], DBSCAN_MAP_TYPES):
    temp = dbscan_map_grid[dbscan_map_grid["권역유형"].eq(region_type)].copy()
    region_count = cluster_gdf[cluster_gdf["권역유형"].eq(region_type)]["취약권역_ID"].nunique()

    ax.scatter(
        temp["중심점_x"],
        temp["중심점_y"],
        color=THEME_ORANGE,
        marker="s",
        s=4.4,
        linewidths=0,
        alpha=0.94,
        rasterized=True,
        zorder=3,
    )
    draw_dbscan_boundary(ax, gu_boundary, seoul_boundary)
    set_dbscan_map_extent(ax, dbscan_map_grid, seoul_boundary)
    ax.set_title(
        f"{region_type} {region_count:,}개 권역",
        fontsize=15,
        color=THEME_DARK,
        pad=10,
        **TITLE_KWARGS,
    )

for ax in axes[len(DBSCAN_MAP_TYPES):]:
    ax.set_facecolor(THEME_BG)
    ax.axis("off")

fig.suptitle(
    "DBSCAN 취약 권역",
    fontsize=27,
    color=THEME_DARK,
    y=0.965,
    **TITLE_KWARGS,
)

dbscan_map_path = DBSCAN_IMAGE_PATH / "dbscan_vulnerable_regions_top10.png"
fig.savefig(dbscan_map_path, bbox_inches="tight", facecolor=THEME_BG)
plt.show()
plt.close(fig)

print("저장:", dbscan_map_path)


## 7. 결과 요약

- DBSCAN은 종합취약지수와 4개 하위지표를 각각 별도의 권역유형으로 처리함.
- 각 권역유형은 관련 인구가 있는 격자만 대상으로 하며, 점수 상위 10% 격자를 권역화함.
- 기본 파라미터는 `eps=150m`, `min_samples=4`로 적용함.
- 고립 취약격자는 별도 파일을 만들지 않고 `DBSCAN_취약격자.csv`에서 `DBSCAN_label=-1`로 확인함.
- 공간파일은 이번 산출에서 제외했으며, 권역면적은 포함 100m 격자 수 × 10,000㎡로 계산함.
- 최종 해석은 권역 수, 고립격자 비율, 권역별 관련 인구와 평균 취약점수를 함께 보고 판단함.


## 8. 공간적 자기상관 심화분석

- 목적: DBSCAN 상위 10% 취약권역이 실제 공간적 취약 밀집을 반영하는지 검증함.
- 기준: 전체 분석대상 격자에서 공간적 자기상관을 계산한 뒤, DBSCAN 권역과 중첩해 확인함.
- 시각화: Moran 산점도 1장, LISA + DBSCAN 중첩 지도 1장만 생성함.


In [ ]:
from scipy.sparse import csr_matrix, diags
from scipy.spatial import cKDTree

import matplotlib
matplotlib.use("Agg")
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

STATIC_FONT_DIR = PROJECT_PATH / "analysis_table" / "image" / "_fonts"
MEDIUM_FONT_PATH = STATIC_FONT_DIR / "NotoSansKR-Medium.ttf"
BOLD_FONT_PATH = STATIC_FONT_DIR / "NotoSansKR-Bold.ttf"
FONT_PATH = Path("C:/Windows/Fonts/NotoSansKR-VF.ttf")

if MEDIUM_FONT_PATH.exists():
    fm.fontManager.addfont(str(MEDIUM_FONT_PATH))
    BODY_FONT = fm.FontProperties(fname=str(MEDIUM_FONT_PATH))
    LEGEND_FONT = fm.FontProperties(fname=str(MEDIUM_FONT_PATH), size=8)
    plt.rcParams["font.family"] = BODY_FONT.get_name()
elif FONT_PATH.exists():
    fm.fontManager.addfont(str(FONT_PATH))
    BODY_FONT = fm.FontProperties(family="Noto Sans KR")
    LEGEND_FONT = fm.FontProperties(family="Noto Sans KR", size=8)
    plt.rcParams["font.family"] = "Noto Sans KR"
else:
    BODY_FONT = None
    LEGEND_FONT = None
    plt.rcParams["font.family"] = "Malgun Gothic"

if BOLD_FONT_PATH.exists():
    fm.fontManager.addfont(str(BOLD_FONT_PATH))
    TITLE_FONT = fm.FontProperties(fname=str(BOLD_FONT_PATH))
elif FONT_PATH.exists():
    TITLE_FONT = fm.FontProperties(family="Noto Sans KR", weight="bold")
else:
    TITLE_FONT = None

TITLE_KWARGS = {"fontproperties": TITLE_FONT} if TITLE_FONT is not None else {"fontweight": "bold"}
BODY_KWARGS = {"fontproperties": BODY_FONT} if BODY_FONT is not None else {}

plt.rcParams.update({
    "axes.unicode_minus": False,
    "axes.titleweight": "bold",
    "figure.titleweight": "bold",
})

THEME_ORANGE = "#ff641d"
THEME_LIGHT_GRAY = "#dedbd5"
THEME_DARK = "#2f2a26"

try:
    import geopandas as gpd
except ImportError:
    gpd = None

SPATIAL_IMAGE_PATH = DASHBOARD_PATH / "IMAGE" / "spatial_autocorrelation"
SPATIAL_IMAGE_PATH.mkdir(parents=True, exist_ok=True)

SPATIAL_NEIGHBOR_DISTANCE = 150
SPATIAL_PERMUTATIONS = 199
SPATIAL_RANDOM_SEED = 42

AUTOCORR_SCORE_CONFIGS = [
    {
        "권역유형": "종합취약",
        "점수컬럼": "최종취약지수",
        "대상조건": final_grid["분석대상여부"],
    },
    {
        "권역유형": "시설접근성취약",
        "점수컬럼": "시설분류_접근성취약도",
        "대상조건": final_grid["분석대상여부"],
    },
    {
        "권역유형": "문화다양성취약",
        "점수컬럼": "문화다양성부족도",
        "대상조건": final_grid["분석대상여부"],
    },
    {
        "권역유형": "장애인친화취약",
        "점수컬럼": "장애인친화시설_접근성취약도",
        "대상조건": final_grid["장애인_수요인구수"].gt(0),
    },
    {
        "권역유형": "노인편의취약",
        "점수컬럼": "노인편의서비스_접근성취약도",
        "대상조건": final_grid["노령인구_수요인구수"].gt(0),
    },
]

LISA_COLOR = {
    "High-High": "#d7191c",
    "Low-Low": "#457b9d",
    "High-Low": "#ff9f1c",
    "Low-High": "#7b2cbf",
    "유의하지 않음": "#dedbd5",
}


def build_distance_weight(coords, threshold):
    tree = cKDTree(coords)
    neighbors = tree.query_ball_point(coords, r=threshold)
    rows = []
    cols = []

    for idx, neighbor_ids in enumerate(neighbors):
        for neighbor_idx in neighbor_ids:
            if idx != neighbor_idx:
                rows.append(idx)
                cols.append(neighbor_idx)

    data = np.ones(len(rows), dtype="float64")
    weight = csr_matrix((data, (rows, cols)), shape=(len(coords), len(coords)))
    raw_neighbor_count = np.asarray(weight.sum(axis=1)).ravel()

    row_sum = raw_neighbor_count.copy()
    row_sum[row_sum == 0] = 1
    weight = diags(1 / row_sum) @ weight

    return weight, raw_neighbor_count


def calculate_moran(values, coords, permutations=199, random_seed=42):
    values = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype="float64")
    valid_mask = np.isfinite(values)
    values = values[valid_mask]
    coords = coords[valid_mask]

    weight, raw_neighbor_count = build_distance_weight(coords, SPATIAL_NEIGHBOR_DISTANCE)
    centered = values - values.mean()
    denominator = centered @ centered
    weight_sum = weight.sum()
    n = len(values)

    spatial_lag = weight.dot(centered)
    moran_i = (n / weight_sum) * (centered @ spatial_lag) / denominator

    rng = np.random.default_rng(random_seed)
    permuted_i = np.empty(permutations, dtype="float64")
    for perm_idx in range(permutations):
        permuted = rng.permutation(centered)
        permuted_i[perm_idx] = (n / weight_sum) * (permuted @ weight.dot(permuted)) / denominator

    p_value = (np.sum(np.abs(permuted_i) >= abs(moran_i)) + 1) / (permutations + 1)
    z_score = (moran_i - permuted_i.mean()) / permuted_i.std(ddof=1)

    return {
        "values": values,
        "valid_mask": valid_mask,
        "weight": weight,
        "raw_neighbor_count": raw_neighbor_count,
        "centered": centered,
        "spatial_lag": spatial_lag,
        "moran_i": moran_i,
        "p_value": p_value,
        "z_score": z_score,
    }


def calculate_lisa(moran_result, permutations=199, random_seed=42):
    values = moran_result["values"]
    weight = moran_result["weight"]
    standardized = (values - values.mean()) / values.std(ddof=0)
    spatial_lag = weight.dot(standardized)
    local_i = standardized * spatial_lag

    rng = np.random.default_rng(random_seed)
    extreme_count = np.zeros(len(values), dtype="int32")
    abs_observed = np.abs(local_i)

    for perm_idx in range(permutations):
        permuted = rng.permutation(standardized)
        permuted_local = standardized * weight.dot(permuted)
        extreme_count += np.abs(permuted_local) >= abs_observed

    local_p = (extreme_count + 1) / (permutations + 1)
    lisa_type = np.full(len(values), "유의하지 않음", dtype=object)
    significant = local_p < 0.05

    lisa_type[significant & (standardized > 0) & (spatial_lag > 0)] = "High-High"
    lisa_type[significant & (standardized < 0) & (spatial_lag < 0)] = "Low-Low"
    lisa_type[significant & (standardized > 0) & (spatial_lag < 0)] = "High-Low"
    lisa_type[significant & (standardized < 0) & (spatial_lag > 0)] = "Low-High"

    return {
        "standardized": standardized,
        "spatial_lag": spatial_lag,
        "local_i": local_i,
        "local_p": local_p,
        "lisa_type": lisa_type,
    }


def interpret_moran(moran_i, p_value):
    if p_value >= 0.05:
        return "유의하지 않음"
    if moran_i > 0:
        return "유의한 공간 밀집"
    if moran_i < 0:
        return "유의한 교차/분산"
    return "패턴 약함"


def load_seoul_boundary():
    boundary_path = PROJECT_PATH / "analysis_table" / "data" / "output" / "서울시_시군구_행정동_경계.gpkg"
    if gpd is None or not boundary_path.exists():
        return None, None

    dong_boundary = gpd.read_file(boundary_path)
    if dong_boundary.crs is not None:
        dong_boundary = dong_boundary.to_crs(epsg=5179)

    gu_boundary = dong_boundary[["시군구", "geometry"]].dissolve(by="시군구", as_index=False)
    seoul_boundary = dong_boundary[["geometry"]].dissolve()
    return gu_boundary, seoul_boundary


def draw_boundary(ax, gu_boundary, seoul_boundary):
    if gu_boundary is None or seoul_boundary is None:
        return
    gu_boundary.boundary.plot(ax=ax, color="#c9c4bc", linewidth=0.35, zorder=1)
    seoul_boundary.boundary.plot(ax=ax, color=THEME_DARK, linewidth=0.85, zorder=4)


def set_map_extent(ax, data, seoul_boundary=None):
    if seoul_boundary is not None:
        minx, miny, maxx, maxy = seoul_boundary.total_bounds
    else:
        minx, maxx = data["중심점_x"].min(), data["중심점_x"].max()
        miny, maxy = data["중심점_y"].min(), data["중심점_y"].max()

    x_pad = (maxx - minx) * 0.025
    y_pad = (maxy - miny) * 0.025
    ax.set_xlim(minx - x_pad, maxx + x_pad)
    ax.set_ylim(miny - y_pad, maxy + y_pad)
    ax.set_aspect("equal")
    ax.axis("off")


### 8-1. Moran's I

- 목적: 취약점수가 전체 공간에서 유의하게 뭉쳐 있는지 확인함.
- 기준: 전체 분석대상 격자에서 150m 이내 이웃 관계를 구성함.
- 시각화: 종합취약지수 Moran scatter plot만 생성함.


In [ ]:
moran_rows = []
moran_result_map = {}
lisa_result_map = {}
lisa_grid_map = {}

for config in AUTOCORR_SCORE_CONFIGS:
    region_type = config["권역유형"]
    score_col = config["점수컬럼"]
    analysis_grid = final_grid[
        config["대상조건"]
        & final_grid[score_col].notna()
        & final_grid[["중심점_x", "중심점_y"]].notna().all(axis=1)
    ].copy()

    coords = analysis_grid[["중심점_x", "중심점_y"]].to_numpy(dtype="float64")
    moran_result = calculate_moran(
        analysis_grid[score_col],
        coords,
        permutations=SPATIAL_PERMUTATIONS,
        random_seed=SPATIAL_RANDOM_SEED,
    )
    lisa_result = calculate_lisa(
        moran_result,
        permutations=SPATIAL_PERMUTATIONS,
        random_seed=SPATIAL_RANDOM_SEED,
    )

    valid_analysis = analysis_grid.iloc[np.where(moran_result["valid_mask"])[0]].copy()
    local_grid = valid_analysis[["GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y", score_col]].copy()
    local_grid["권역유형"] = region_type
    local_grid["local_moran_i"] = lisa_result["local_i"]
    local_grid["local_p_value"] = lisa_result["local_p"]
    local_grid["LISA유형"] = lisa_result["lisa_type"]

    moran_result_map[region_type] = moran_result
    lisa_result_map[region_type] = lisa_result
    lisa_grid_map[region_type] = local_grid

    neighbor_count = moran_result["raw_neighbor_count"]
    moran_rows.append({
        "권역유형": region_type,
        "분석격자수": len(valid_analysis),
        "평균이웃수": neighbor_count.mean(),
        "중앙이웃수": np.median(neighbor_count),
        "고립격자수": int((neighbor_count == 0).sum()),
        "Moran_I": moran_result["moran_i"],
        "z_score": moran_result["z_score"],
        "p_value": moran_result["p_value"],
        "해석": interpret_moran(moran_result["moran_i"], moran_result["p_value"]),
    })

moran_summary = pd.DataFrame(moran_rows)

print("Moran's I 분석 기준")
print(f"- 공간 이웃 기준: 중심점 {SPATIAL_NEIGHBOR_DISTANCE}m 이내")
print(f"- permutation: {SPATIAL_PERMUTATIONS}회")
print()
print("Moran's I 지표별 결과")
display(
    moran_summary.round({
        "평균이웃수": 2,
        "중앙이웃수": 2,
        "Moran_I": 4,
        "z_score": 2,
        "p_value": 4,
    })
)

main_type = "종합취약"
main_moran = moran_result_map[main_type]
main_lisa = lisa_result_map[main_type]
plot_data = pd.DataFrame({
    "표준화_취약점수": main_lisa["standardized"],
    "주변평균_표준화취약점수": main_lisa["spatial_lag"],
})

fig, ax = plt.subplots(figsize=(6.4, 5.2), dpi=160)
ax.scatter(
    plot_data["표준화_취약점수"],
    plot_data["주변평균_표준화취약점수"],
    s=5,
    color=THEME_LIGHT_GRAY,
    alpha=0.55,
    linewidths=0,
    rasterized=True,
)
coef = np.polyfit(plot_data["표준화_취약점수"], plot_data["주변평균_표준화취약점수"], 1)
x_line = np.linspace(plot_data["표준화_취약점수"].min(), plot_data["표준화_취약점수"].max(), 100)
ax.plot(x_line, coef[0] * x_line + coef[1], color=THEME_ORANGE, linewidth=2.4)
ax.axhline(0, color="#b9aa9d", linewidth=0.9)
ax.axvline(0, color="#b9aa9d", linewidth=0.9)
ax.set_title(
    f"종합취약 Moran scatter plot · I={main_moran['moran_i']:.4f}, p={main_moran['p_value']:.3f}",
    fontsize=16,
    color=THEME_DARK,
    pad=12,
    **TITLE_KWARGS,
)
ax.set_xlabel("표준화 취약점수", fontsize=11, **BODY_KWARGS)
ax.set_ylabel("주변 평균 표준화 취약점수", fontsize=11, **BODY_KWARGS)
ax.grid(color="#eee7df", linewidth=0.8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#b9aa9d")
ax.spines["bottom"].set_color("#b9aa9d")
ax.tick_params(colors=THEME_DARK)

moran_scatter_path = SPATIAL_IMAGE_PATH / "moran_scatter_종합취약.png"
fig.savefig(moran_scatter_path, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

print("저장:", moran_scatter_path)


### 8-2. LISA

- 목적: 어느 격자가 실제 취약 핫스팟인지 확인하고 DBSCAN 권역과 중첩해 검증함.
- 기준: 전체 분석대상 격자에서 LISA 유형을 산출한 뒤, 상위 10% DBSCAN 권역 내부의 High-High 비율을 확인함.
- 시각화: 종합취약 LISA 지도와 DBSCAN-High-High 중첩 지도를 한 장으로 생성함.


In [ ]:
lisa_summary_rows = []
validation_rows = []

for config in AUTOCORR_SCORE_CONFIGS:
    region_type = config["권역유형"]
    local_grid = lisa_grid_map[region_type]
    lisa_counts = local_grid["LISA유형"].value_counts()

    lisa_summary_rows.append({
        "권역유형": region_type,
        "High-High": int(lisa_counts.get("High-High", 0)),
        "Low-Low": int(lisa_counts.get("Low-Low", 0)),
        "High-Low": int(lisa_counts.get("High-Low", 0)),
        "Low-High": int(lisa_counts.get("Low-High", 0)),
        "유의하지않음": int(lisa_counts.get("유의하지 않음", 0)),
        "유의격자비율": (local_grid["LISA유형"].ne("유의하지 않음").mean() * 100),
        "High-High비율": (local_grid["LISA유형"].eq("High-High").mean() * 100),
    })

    dbscan_region_grid = vulnerable_grid[
        vulnerable_grid["권역유형"].eq(region_type)
        & vulnerable_grid["DBSCAN_label"].ge(0)
    ].copy()
    overlap = dbscan_region_grid[["GRID_CD", "취약권역_ID", "중심점_x", "중심점_y"]].merge(
        local_grid[["GRID_CD", "LISA유형", "local_moran_i", "local_p_value"]],
        on="GRID_CD",
        how="left",
    )

    total_count = len(overlap)
    high_high_count = int(overlap["LISA유형"].eq("High-High").sum())
    significant_count = int(overlap["LISA유형"].ne("유의하지 않음").sum())

    validation_rows.append({
        "권역유형": region_type,
        "취약권역수": int(overlap["취약권역_ID"].nunique()),
        "권역포함격자수": total_count,
        "유의LISA격자수": significant_count,
        "유의LISA비율": significant_count / total_count * 100 if total_count > 0 else 0,
        "High-High격자수": high_high_count,
        "High-High비율": high_high_count / total_count * 100 if total_count > 0 else 0,
    })

lisa_summary = pd.DataFrame(lisa_summary_rows)
lisa_dbscan_validation = pd.DataFrame(validation_rows)

print("LISA 유형별 격자 수")
display(lisa_summary.round({"유의격자비율": 2, "High-High비율": 2}))

print("DBSCAN 상위 10% 권역과 LISA High-High 중첩")
display(lisa_dbscan_validation.round({"유의LISA비율": 2, "High-High비율": 2}))

gu_boundary, seoul_boundary = load_seoul_boundary()
main_lisa_grid = lisa_grid_map["종합취약"].copy()
main_dbscan_grid = vulnerable_grid[
    vulnerable_grid["권역유형"].eq("종합취약")
    & vulnerable_grid["DBSCAN_label"].ge(0)
].copy()
main_overlap = main_dbscan_grid[["GRID_CD", "취약권역_ID", "중심점_x", "중심점_y"]].merge(
    main_lisa_grid[["GRID_CD", "LISA유형"]],
    on="GRID_CD",
    how="left",
)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.8), dpi=170)

for lisa_type in ["유의하지 않음", "Low-Low", "Low-High", "High-Low", "High-High"]:
    temp = main_lisa_grid[main_lisa_grid["LISA유형"].eq(lisa_type)]
    axes[0].scatter(
        temp["중심점_x"],
        temp["중심점_y"],
        color=LISA_COLOR[lisa_type],
        s=3.2 if lisa_type != "유의하지 않음" else 2.0,
        linewidths=0,
        alpha=0.85 if lisa_type != "유의하지 않음" else 0.35,
        label=lisa_type,
        rasterized=True,
        zorder=3 if lisa_type != "유의하지 않음" else 2,
    )
draw_boundary(axes[0], gu_boundary, seoul_boundary)
axes[0].set_title("종합취약 LISA cluster map", fontsize=15, color=THEME_DARK, pad=10, **TITLE_KWARGS)
set_map_extent(axes[0], main_lisa_grid, seoul_boundary)

base_overlap = main_overlap[main_overlap["LISA유형"].ne("High-High")]
hh_overlap = main_overlap[main_overlap["LISA유형"].eq("High-High")]
axes[1].scatter(
    base_overlap["중심점_x"],
    base_overlap["중심점_y"],
    color="#ffb36b",
    s=5,
    linewidths=0,
    alpha=0.65,
    label="DBSCAN 권역",
    rasterized=True,
    zorder=2,
)
axes[1].scatter(
    hh_overlap["중심점_x"],
    hh_overlap["중심점_y"],
    color=LISA_COLOR["High-High"],
    s=5.5,
    linewidths=0,
    alpha=0.9,
    label="DBSCAN + High-High",
    rasterized=True,
    zorder=3,
)
draw_boundary(axes[1], gu_boundary, seoul_boundary)
axes[1].set_title("DBSCAN 상위 10% + LISA High-High", fontsize=15, color=THEME_DARK, pad=10, **TITLE_KWARGS)
set_map_extent(axes[1], main_lisa_grid, seoul_boundary)

handles, labels = axes[0].get_legend_handles_labels()
legend_kwargs = {"prop": LEGEND_FONT} if LEGEND_FONT is not None else {"fontsize": 7.5}
axes[0].legend(handles, labels, loc="lower left", frameon=True, markerscale=2.2, **legend_kwargs)
axes[1].legend(loc="lower left", frameon=True, markerscale=2.2, **legend_kwargs)
fig.suptitle("종합취약 LISA 및 DBSCAN 권역 검증", fontsize=20, color=THEME_DARK, y=0.985, **TITLE_KWARGS)
fig.tight_layout(rect=[0, 0, 1, 0.94])

lisa_map_path = SPATIAL_IMAGE_PATH / "lisa_dbscan_highhigh_종합취약.png"
fig.savefig(lisa_map_path, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

print("저장:", lisa_map_path)


## 9. DBSCAN 취약권역 프로파일링

- 목적: DBSCAN 취약권역별로 행정동, 인구, 장애인·노인 수요, 도달가능 가맹점수 등 정책 해석용 속성을 붙임.
- 입력: `DBSCAN_취약권역.csv`, `DBSCAN_취약격자.csv`, `격자별_최종취약지수.csv`
- 저장: `DBSCAN_취약권역_프로파일.csv`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

if "PROJECT_PATH" not in globals():
    BASE_PATH = Path.cwd().resolve()
    if BASE_PATH.name == "dashboard":
        PROJECT_PATH = BASE_PATH.parents[1]
    elif BASE_PATH.name == "notebooks":
        PROJECT_PATH = BASE_PATH.parent
    elif (BASE_PATH / "notebooks").exists():
        PROJECT_PATH = BASE_PATH
    else:
        PROJECT_PATH = BASE_PATH

if "OUTPUT_PATH" not in globals():
    OUTPUT_PATH = PROJECT_PATH / "notebooks" / "dashboard" / "OUTPUT" / "vulnerability_index"

region_path = OUTPUT_PATH / "DBSCAN_취약권역.csv"
vulnerable_grid_path = OUTPUT_PATH / "DBSCAN_취약격자.csv"
final_grid_path = OUTPUT_PATH / "격자별_최종취약지수.csv"
profile_path = OUTPUT_PATH / "DBSCAN_취약권역_프로파일.csv"

region_base = pd.read_csv(region_path, encoding="utf-8-sig")
dbscan_grid = pd.read_csv(vulnerable_grid_path, encoding="utf-8-sig")
full_grid = pd.read_csv(final_grid_path, encoding="utf-8-sig")

profile_source_cols = [
    "GRID_CD", "시군구", "행정동", "추정_인구수", "문화누리대상자_추정인구수",
    "장애인_수요인구수", "노령인구_수요인구수", "reachable_store_n",
    "접근가능중분류수", "평균접근가능가맹점수", "시설접근성_최취약중분류",
    "장애인친화시설_통합도달가맹점수", "장애인친화시설_도달가능중분류", "장애인친화시설_부족중분류",
    "노인편의서비스_통합도달가맹점수", "노인편의서비스_도달가능중분류", "노인편의서비스_부족중분류",
]
profile_source_cols = [col for col in profile_source_cols if col in full_grid.columns]

profile_grid = (
    dbscan_grid.loc[dbscan_grid["DBSCAN_label"].ge(0), ["권역유형", "취약권역_ID", "GRID_CD"]]
    .merge(full_grid[profile_source_cols], on="GRID_CD", how="left", validate="many_to_one")
)

numeric_cols = [
    "추정_인구수", "문화누리대상자_추정인구수", "장애인_수요인구수", "노령인구_수요인구수",
    "reachable_store_n", "접근가능중분류수", "평균접근가능가맹점수",
    "장애인친화시설_통합도달가맹점수", "노인편의서비스_통합도달가맹점수",
]
for col in numeric_cols:
    if col in profile_grid.columns:
        profile_grid[col] = pd.to_numeric(profile_grid[col], errors="coerce").fillna(0)


def top_values(series, n=5):
    values = series.dropna().astype(str).str.strip()
    values = values[(values != "") & (values != "nan")]
    if values.empty:
        return ""
    return ", ".join(values.value_counts().head(n).index.tolist())


def top_split_values(series, n=5):
    values = []
    for item in series.dropna().astype(str):
        for part in item.split(","):
            part = part.strip()
            if part and part != "nan":
                values.append(part)
    if not values:
        return ""
    return ", ".join(pd.Series(values).value_counts().head(n).index.tolist())


def safe_sum(series):
    return pd.to_numeric(series, errors="coerce").fillna(0).sum()


def safe_mean(series):
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).mean()


def safe_min(series):
    clean = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return clean.min()


def safe_max(series):
    clean = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return clean.max()

profile_rows = []
for (area_type, region_id), group in profile_grid.groupby(["권역유형", "취약권역_ID"], dropna=False):
    row = {
        "권역유형": area_type,
        "취약권역_ID": region_id,
        "포함_시군구수": group["시군구"].nunique(dropna=True) if "시군구" in group else np.nan,
        "포함_행정동수": group["행정동"].nunique(dropna=True) if "행정동" in group else np.nan,
        "포함_행정동목록": top_values(group["행정동"], n=6) if "행정동" in group else "",
        "총추정인구수": safe_sum(group.get("추정_인구수", pd.Series(dtype=float))),
        "문화누리대상자수": safe_sum(group.get("문화누리대상자_추정인구수", pd.Series(dtype=float))),
        "장애인인구수": safe_sum(group.get("장애인_수요인구수", pd.Series(dtype=float))),
        "노인인구수": safe_sum(group.get("노령인구_수요인구수", pd.Series(dtype=float))),
        "평균_도달가능가맹점수": safe_mean(group.get("reachable_store_n", pd.Series(dtype=float))),
        "최소_도달가능가맹점수": safe_min(group.get("reachable_store_n", pd.Series(dtype=float))),
        "최대_도달가능가맹점수": safe_max(group.get("reachable_store_n", pd.Series(dtype=float))),
        "평균_접근가능중분류수": safe_mean(group.get("접근가능중분류수", pd.Series(dtype=float))),
        "평균_격자당중분류별가맹점수": safe_mean(group.get("평균접근가능가맹점수", pd.Series(dtype=float))),
        "장애인친화_평균도달가맹점수": safe_mean(group.get("장애인친화시설_통합도달가맹점수", pd.Series(dtype=float))),
        "노인편의_평균도달가맹점수": safe_mean(group.get("노인편의서비스_통합도달가맹점수", pd.Series(dtype=float))),
        "주요_시설접근성취약중분류": top_values(group.get("시설접근성_최취약중분류", pd.Series(dtype=object)), n=5),
        "장애인친화_주요부족중분류": top_split_values(group.get("장애인친화시설_부족중분류", pd.Series(dtype=object)), n=5),
        "노인편의_주요부족중분류": top_split_values(group.get("노인편의서비스_부족중분류", pd.Series(dtype=object)), n=5),
    }
    profile_rows.append(row)

region_profile_attrs = pd.DataFrame(profile_rows)

base_cols = [
    "권역유형", "취약권역_ID", "취약권역등급", "주요_시군구", "주요_행정동",
    "포함_취약격자수", "권역면적_m2", "권역중심_x", "권역중심_y",
    "기준점수컬럼", "취약선정기준", "평균_기준취약점수", "최고_기준취약점수",
    "수요가중_기준취약점수", "평균_최종취약지수", "수요가중_최종취약지수",
    "평균_우선지원지수", "주요취약원인_1", "주요취약원인_2",
]
base_cols = [col for col in base_cols if col in region_base.columns]

region_profile = (
    region_base[base_cols]
    .merge(region_profile_attrs, on=["권역유형", "취약권역_ID"], how="left", validate="one_to_one")
)

sort_cols = ["권역유형", "취약권역등급", "수요가중_기준취약점수", "포함_취약격자수"]
sort_cols = [col for col in sort_cols if col in region_profile.columns]
region_profile = region_profile.sort_values(
    sort_cols,
    ascending=[True, True, False, False][:len(sort_cols)]
).copy()

round_cols = [
    "평균_기준취약점수", "최고_기준취약점수", "수요가중_기준취약점수",
    "평균_최종취약지수", "수요가중_최종취약지수", "평균_우선지원지수",
    "평균_도달가능가맹점수", "최소_도달가능가맹점수", "최대_도달가능가맹점수",
    "평균_접근가능중분류수", "평균_격자당중분류별가맹점수",
    "장애인친화_평균도달가맹점수", "노인편의_평균도달가맹점수",
]
for col in round_cols:
    if col in region_profile.columns:
        region_profile[col] = region_profile[col].round(2)

count_cols = [
    "총추정인구수", "문화누리대상자수", "장애인인구수", "노인인구수",
    "포함_시군구수", "포함_행정동수",
]
for col in count_cols:
    if col in region_profile.columns:
        region_profile[col] = region_profile[col].round(0).astype("Int64")

region_profile.to_csv(profile_path, index=False, encoding="utf-8-sig")

print("저장:", profile_path)
print("권역 프로파일 shape:", region_profile.shape)
print()
print("종합취약 권역 프로파일 예시")
display(
    region_profile[region_profile["권역유형"].eq("종합취약")]
    .head(10)
)


## 10. DBSCAN 취약권역 프로파일 시각화

- 목적: 권역 프로파일링 결과를 발표용 표와 그래프로 정리함.
- 기준: 종합취약 권역을 중심으로 등급별 특성, 수요-도달가능 가맹점 관계, 우선 검토 권역을 시각화함.
- 저장 위치: `notebooks/dashboard/IMAGE/vulnerability_index/profile/`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

try:
    from IPython.display import display, Image as IPyImage
except Exception:
    display = None
    IPyImage = None

if "PROJECT_PATH" not in globals():
    BASE_PATH = Path.cwd().resolve()
    if BASE_PATH.name == "dashboard":
        PROJECT_PATH = BASE_PATH.parents[1]
    elif BASE_PATH.name == "notebooks":
        PROJECT_PATH = BASE_PATH.parent
    elif (BASE_PATH / "notebooks").exists():
        PROJECT_PATH = BASE_PATH
    else:
        PROJECT_PATH = BASE_PATH

DASHBOARD_PATH = PROJECT_PATH / "notebooks" / "dashboard"
OUTPUT_PATH = DASHBOARD_PATH / "OUTPUT" / "vulnerability_index"
PROFILE_IMAGE_PATH = DASHBOARD_PATH / "IMAGE" / "vulnerability_index" / "profile"
PROFILE_IMAGE_PATH.mkdir(parents=True, exist_ok=True)

STATIC_FONT_DIR = PROJECT_PATH / "analysis_table" / "image" / "_fonts"
MEDIUM_FONT_PATH = STATIC_FONT_DIR / "NotoSansKR-Medium.ttf"
BOLD_FONT_PATH = STATIC_FONT_DIR / "NotoSansKR-Bold.ttf"
FONT_PATH = Path("C:/Windows/Fonts/NotoSansKR-VF.ttf")

if MEDIUM_FONT_PATH.exists():
    fm.fontManager.addfont(str(MEDIUM_FONT_PATH))
    BODY_FONT = fm.FontProperties(fname=str(MEDIUM_FONT_PATH))
    plt.rcParams["font.family"] = BODY_FONT.get_name()
elif FONT_PATH.exists():
    fm.fontManager.addfont(str(FONT_PATH))
    BODY_FONT = fm.FontProperties(family="Noto Sans KR")
    plt.rcParams["font.family"] = "Noto Sans KR"
else:
    BODY_FONT = None
    plt.rcParams["font.family"] = "Malgun Gothic"

if BOLD_FONT_PATH.exists():
    fm.fontManager.addfont(str(BOLD_FONT_PATH))
    TITLE_FONT = fm.FontProperties(fname=str(BOLD_FONT_PATH))
elif FONT_PATH.exists():
    TITLE_FONT = fm.FontProperties(family="Noto Sans KR", weight="bold")
else:
    TITLE_FONT = None

TITLE_KWARGS = {"fontproperties": TITLE_FONT} if TITLE_FONT is not None else {"fontweight": "bold"}
BODY_KWARGS = {"fontproperties": BODY_FONT} if BODY_FONT is not None else {}

BG = "#FBF6EF"
TEXT = "#2A211D"
MUTED = "#746C66"
AXIS = "#D8D2CA"
GREY = "#C9C3BE"
PALE = "#E8DED5"
ORANGE = "#F46B2F"
TEAL = "#138C86"

plt.rcParams.update({
    "axes.unicode_minus": False,
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": TEXT,
    "axes.labelcolor": TEXT,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
})

profile_path = OUTPUT_PATH / "DBSCAN_취약권역_프로파일.csv"
profile = pd.read_csv(profile_path, encoding="utf-8-sig")
composite = profile[profile["권역유형"].eq("종합취약")].copy()

for col in [
    "포함_취약격자수", "총추정인구수", "문화누리대상자수", "장애인인구수", "노인인구수",
    "평균_도달가능가맹점수", "평균_접근가능중분류수", "평균_기준취약점수", "평균_우선지원지수"
]:
    composite[col] = pd.to_numeric(composite[col], errors="coerce").fillna(0)

GRADE_ORDER = ["최우선취약권역", "우선취약권역", "일반취약권역"]
GRADE_LABEL = {
    "최우선취약권역": "최우선",
    "우선취약권역": "우선",
    "일반취약권역": "일반",
}
GRADE_COLOR = {
    "최우선취약권역": ORANGE,
    "우선취약권역": TEAL,
    "일반취약권역": GREY,
}
CAUSE_SHORT = {
    "시설분류 접근성 부족": "시설접근성",
    "문화시설 다양성 부족": "문화다양성",
    "장애인친화시설 접근성 부족": "장애인친화",
    "노인편의서비스 접근성 부족": "노인편의",
}


def savefig(fig, path):
    fig.savefig(path, dpi=240, bbox_inches="tight", facecolor=BG, pad_inches=0.12)
    plt.close(fig)
    if display is not None and IPyImage is not None:
        display(IPyImage(filename=str(path)))


def style_axis(ax, keep_left=True):
    ax.set_facecolor(BG)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(keep_left)
    if keep_left:
        ax.spines["left"].set_color(AXIS)
        ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_color(AXIS)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(labelsize=10, length=3, color=AXIS)


def fmt_int(value):
    return f"{int(round(value)):,}"


def short_cause(a, b=None):
    first = CAUSE_SHORT.get(str(a), str(a).replace(" 부족", ""))
    if b is None or pd.isna(b):
        return first
    second = CAUSE_SHORT.get(str(b), str(b).replace(" 부족", ""))
    return f"{first}/{second}"

# 0. 권역유형별 프로파일 요약표
TYPE_ORDER = ["종합취약", "시설접근성취약", "문화다양성취약", "노인편의취약", "장애인친화취약"]
type_summary = (
    profile
    .groupby("권역유형")
    .agg(
        권역수=("취약권역_ID", "count"),
        취약격자=("포함_취약격자수", "sum"),
        문화누리=("문화누리대상자수", "sum"),
        장애인=("장애인인구수", "sum"),
        노인=("노인인구수", "sum"),
        평균도달가맹점=("평균_도달가능가맹점수", "mean"),
        평균중분류=("평균_접근가능중분류수", "mean"),
    )
    .reindex([t for t in TYPE_ORDER if t in profile["권역유형"].unique()])
    .reset_index()
)

type_table = type_summary.copy()
type_table["권역수"] = type_table["권역수"].map(lambda x: f"{int(x):,}")
type_table["취약격자"] = type_table["취약격자"].map(lambda x: f"{int(round(x)):,}")
type_table["문화누리"] = type_table["문화누리"].map(lambda x: f"{int(round(x)):,}")
type_table["장애인"] = type_table["장애인"].map(lambda x: f"{int(round(x)):,}")
type_table["노인"] = type_table["노인"].map(lambda x: f"{int(round(x)):,}")
type_table["평균도달가맹점"] = type_table["평균도달가맹점"].map(lambda x: f"{x:.1f}")
type_table["평균중분류"] = type_table["평균중분류"].map(lambda x: f"{x:.2f}")

type_table = type_table.rename(columns={
    "권역유형": "유형",
    "평균도달가맹점": "도달가맹점",
    "평균중분류": "중분류",
})

fig, ax = plt.subplots(figsize=(12.6, 4.3), facecolor=BG)
ax.axis("off")
ax.set_title("취약권역 유형별 프로파일 요약", fontsize=23, color=TEXT, pad=18, **TITLE_KWARGS)

col_widths = [0.20, 0.09, 0.10, 0.12, 0.10, 0.10, 0.13, 0.10]
table = ax.table(
    cellText=type_table.values,
    colLabels=type_table.columns,
    cellLoc="center",
    colLoc="center",
    colWidths=col_widths,
    bbox=[0.00, 0.00, 1.00, 0.82],
)
table.auto_set_font_size(False)
table.set_fontsize(11.0)
table.scale(1, 1.45)

for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor(AXIS)
    cell.set_linewidth(0.55)
    cell.set_facecolor(BG)
    cell.get_text().set_color(TEXT)
    if BODY_FONT is not None:
        cell.get_text().set_fontproperties(BODY_FONT)
    if r == 0:
        cell.set_facecolor(PALE)
        if TITLE_FONT is not None:
            cell.get_text().set_fontproperties(TITLE_FONT)
        else:
            cell.get_text().set_fontweight("bold")
    if c == 0 and r > 0:
        cell.get_text().set_ha("left")
    if c >= 1 and r > 0:
        cell.get_text().set_ha("right")

type_summary_path = PROFILE_IMAGE_PATH / "dbscan_profile_type_summary_table.png"
savefig(fig, type_summary_path)

# 1. 등급별 프로파일 요약
summary = (
    composite
    .groupby("취약권역등급")
    .agg(
        권역수=("취약권역_ID", "count"),
        포함격자수=("포함_취약격자수", "sum"),
        문화누리대상자수=("문화누리대상자수", "sum"),
        평균도달가맹점=("평균_도달가능가맹점수", "mean"),
        평균접근가능중분류=("평균_접근가능중분류수", "mean"),
    )
    .reindex(GRADE_ORDER)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.9), facecolor=BG)
fig.suptitle("종합취약 권역 등급별 프로파일", fontsize=22, color=TEXT, y=1.04, **TITLE_KWARGS)

x = np.arange(len(summary))
labels = [f"{GRADE_LABEL[g]}\n{int(n)}개" for g, n in zip(summary["취약권역등급"], summary["권역수"])]
colors = [GRADE_COLOR[g] for g in summary["취약권역등급"]]

bars = axes[0].bar(x, summary["평균도달가맹점"], color=colors, width=0.58)
style_axis(axes[0])
axes[0].set_xticks(x, labels, **BODY_KWARGS)
axes[0].set_ylabel("평균 도달가능 가맹점 수", fontsize=11, **BODY_KWARGS)
axes[0].set_title("가맹점 접근", fontsize=15, color=TEXT, pad=12, **TITLE_KWARGS)
axes[0].set_ylim(0, summary["평균도달가맹점"].max() * 1.22)
for bar, value in zip(bars, summary["평균도달가맹점"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, value + 3, f"{value:.1f}", ha="center", va="bottom", fontsize=10, color=TEXT, **BODY_KWARGS)

bars = axes[1].bar(x, summary["평균접근가능중분류"], color=colors, width=0.58)
style_axis(axes[1])
axes[1].set_xticks(x, labels, **BODY_KWARGS)
axes[1].set_ylabel("평균 접근가능 중분류 수", fontsize=11, **BODY_KWARGS)
axes[1].set_title("문화 선택지", fontsize=15, color=TEXT, pad=12, **TITLE_KWARGS)
axes[1].set_ylim(0, summary["평균접근가능중분류"].max() * 1.26)
for bar, value in zip(bars, summary["평균접근가능중분류"]):
    axes[1].text(bar.get_x() + bar.get_width()/2, value + 0.08, f"{value:.2f}", ha="center", va="bottom", fontsize=10, color=TEXT, **BODY_KWARGS)

fig.tight_layout()
grade_summary_path = PROFILE_IMAGE_PATH / "dbscan_profile_grade_summary.png"
savefig(fig, grade_summary_path)

# 2. 수요-도달가능 가맹점 산점도
fig, ax = plt.subplots(figsize=(10.8, 6.4), facecolor=BG)
style_axis(ax)
ax.set_title("종합취약 권역 수요-도달가능 가맹점 프로파일", fontsize=21, color=TEXT, pad=18, **TITLE_KWARGS)
ax.set_xlabel("평균 도달가능 가맹점 수", fontsize=11, **BODY_KWARGS)
ax.set_ylabel("문화누리대상자 수", fontsize=11, **BODY_KWARGS)

size_min, size_max = composite["포함_취약격자수"].min(), composite["포함_취약격자수"].max()
if size_max > size_min:
    composite["bubble_size"] = 42 + (np.sqrt(composite["포함_취약격자수"]) - np.sqrt(size_min)) / (np.sqrt(size_max) - np.sqrt(size_min)) * 430
else:
    composite["bubble_size"] = 120

for grade in GRADE_ORDER:
    temp = composite[composite["취약권역등급"].eq(grade)]
    ax.scatter(
        temp["평균_도달가능가맹점수"],
        temp["문화누리대상자수"],
        s=temp["bubble_size"],
        color=GRADE_COLOR[grade],
        alpha=0.78 if grade != "일반취약권역" else 0.45,
        linewidths=0,
        label=GRADE_LABEL[grade],
    )

x_ref = composite["평균_도달가능가맹점수"].median()
y_ref = composite["문화누리대상자수"].median()
ax.axvline(x_ref, color=AXIS, linewidth=1.0)
ax.axhline(y_ref, color=AXIS, linewidth=1.0)

# PPT 캡처 시 겹침을 줄이기 위해 문화누리대상자 수가 큰 핵심 권역만 표기
label_candidates = composite.nlargest(5, "문화누리대상자수").copy()
label_offsets = [(7, 12), (-8, 14), (7, -20), (-8, -14), (7, 16)]

for i, (_, row) in enumerate(label_candidates.iterrows()):
    x0 = row["평균_도달가능가맹점수"]
    y0 = row["문화누리대상자수"]
    dx, dy = label_offsets[i % len(label_offsets)]
    if x0 < 45:
        dx, ha = 7, "left"
    elif x0 > 280:
        dx, ha = -8, "right"
    else:
        ha = "left" if dx > 0 else "right"
    label = f"{row['주요_행정동']} {fmt_int(row['문화누리대상자수'])}명"
    ax.annotate(
        label,
        xy=(x0, y0),
        xytext=(dx, dy),
        textcoords="offset points",
        ha=ha,
        va="bottom" if dy >= 0 else "top",
        fontsize=9.5,
        color=TEXT,
        **BODY_KWARGS,
    )

ax.legend(frameon=False, loc="upper right", prop=BODY_FONT if BODY_FONT is not None else None)
ax.set_xlim(left=-5)
ax.set_ylim(bottom=-40)
scatter_path = PROFILE_IMAGE_PATH / "dbscan_profile_demand_access_scatter.png"
savefig(fig, scatter_path)

# 3. 도달가능 가맹점수 낮은 권역 TOP10
low_access = composite.nsmallest(10, "평균_도달가능가맹점수").sort_values("평균_도달가능가맹점수", ascending=True).copy()
low_access["label"] = low_access["주요_시군구"] + " " + low_access["주요_행정동"] + "\n" + low_access["취약권역_ID"]

fig, ax = plt.subplots(figsize=(10.8, 6.0), facecolor=BG)
style_axis(ax)
bars = ax.barh(
    np.arange(len(low_access)),
    low_access["평균_도달가능가맹점수"],
    color=[GRADE_COLOR.get(g, GREY) for g in low_access["취약권역등급"]],
    height=0.58,
)
ax.set_yticks(np.arange(len(low_access)), low_access["label"], fontsize=10, **BODY_KWARGS)
ax.invert_yaxis()
ax.set_xlabel("평균 도달가능 가맹점 수", fontsize=11, **BODY_KWARGS)
ax.set_title("도달가능 가맹점수가 낮은 종합취약 권역", fontsize=21, color=TEXT, pad=18, **TITLE_KWARGS)
ax.set_xlim(0, max(10, low_access["평균_도달가능가맹점수"].max() * 1.28))
for bar, (_, row) in zip(bars, low_access.iterrows()):
    value = row["평균_도달가능가맹점수"]
    label = f"{value:.1f} | 대상자 {fmt_int(row['문화누리대상자수'])}명"
    ax.text(value + 0.25, bar.get_y() + bar.get_height()/2, label, ha="left", va="center", fontsize=9.5, color=TEXT, **BODY_KWARGS)
low_access_path = PROFILE_IMAGE_PATH / "dbscan_profile_low_access_top10.png"
savefig(fig, low_access_path)

# 4. 우선지원지수 TOP10 표
priority = composite.sort_values(["평균_우선지원지수", "문화누리대상자수"], ascending=[False, False]).head(10).copy()
table_df = pd.DataFrame({
    "권역": priority["취약권역_ID"],
    "행정동": priority["주요_시군구"] + " " + priority["주요_행정동"],
    "등급": priority["취약권역등급"].map(GRADE_LABEL),
    "문화누리": priority["문화누리대상자수"].map(fmt_int),
    "장애인": priority["장애인인구수"].map(fmt_int),
    "노인": priority["노인인구수"].map(fmt_int),
    "도달가맹점": priority["평균_도달가능가맹점수"].map(lambda x: f"{x:.1f}"),
    "중분류": priority["평균_접근가능중분류수"].map(lambda x: f"{x:.1f}"),
    "주요원인": [short_cause(a, b) for a, b in zip(priority["주요취약원인_1"], priority["주요취약원인_2"])],
})

fig, ax = plt.subplots(figsize=(14.4, 6.1), facecolor=BG)
ax.axis("off")
ax.set_title("종합취약 권역 프로파일 TOP 10", fontsize=23, color=TEXT, pad=18, **TITLE_KWARGS)

col_widths = [0.08, 0.17, 0.08, 0.10, 0.08, 0.08, 0.11, 0.08, 0.22]
table = ax.table(
    cellText=table_df.values,
    colLabels=table_df.columns,
    cellLoc="center",
    colLoc="center",
    colWidths=col_widths,
    bbox=[0.00, 0.00, 1.00, 0.88],
)
table.auto_set_font_size(False)
table.set_fontsize(10.5)
table.scale(1, 1.55)

for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor(AXIS)
    cell.set_linewidth(0.55)
    cell.set_facecolor(BG)
    cell.get_text().set_color(TEXT)
    if BODY_FONT is not None:
        cell.get_text().set_fontproperties(BODY_FONT)
    if r == 0:
        cell.set_facecolor(PALE)
        if TITLE_FONT is not None:
            cell.get_text().set_fontproperties(TITLE_FONT)
        else:
            cell.get_text().set_fontweight("bold")
    if c in [3, 4, 5, 6, 7] and r > 0:
        cell.get_text().set_ha("right")
    if c in [1, 8] and r > 0:
        cell.get_text().set_ha("left")

priority_table_path = PROFILE_IMAGE_PATH / "dbscan_profile_priority_top10_table.png"
savefig(fig, priority_table_path)

print("저장 완료")
for path in [type_summary_path, grade_summary_path, scatter_path, low_access_path, priority_table_path]:
    print(path)
